# Práctica guiada: del modelo legado al artefacto explícito

En la semana 3 usamos un payload **joblib** con tres claves: el estimador, los nombres de las features y la versión del modelo. En esta práctica vamos a separar responsabilidades:

- **model.joblib**: el objeto ejecutable;
- **manifest.json**: el contrato que permite saber si el objeto es compatible;
- código y pruebas: las reglas de validación y el comportamiento ante errores.

## Resultado esperado

Al terminar tendrás un manifiesto mínimo escrito como JSON, una lista de invariantes y una matriz de fallos que otra persona podrá implementar en la segunda clase.

## 1. Inspeccionar el punto de partida

Primero comprueba qué columnas recibe el módulo de inferencia de la semana 3. No cambies todavía el modelo ni ejecutes predicciones.

In [ ]:
from pathlib import Path
import csv

week_root = next(
    parent
    for parent in (Path.cwd(), *Path.cwd().parents)
    if (parent / "modules/04-model-packaging").is_dir()
)
input_path = week_root / "assets/03-wine-quality/inference_samples.csv"
if not input_path.exists():
    input_path = week_root.parent / "semana3/assets/03-wine-quality/inference_samples.csv"

with input_path.open(newline="", encoding="utf-8") as file:
    reader = csv.DictReader(file)
    rows = list(reader)
print(f"Fichero: {input_path}")
print(f"Columnas: {reader.fieldnames}")
print(f"Filas de ejemplo: {len(rows)}")

## 2. Diseñar la frontera del artefacto

Completa esta tabla antes de escribir código. La pregunta clave es: **¿qué necesita un proceso de inferencia para decidir que puede utilizar el artefacto?**

| Elemento | ¿Dónde vive? | ¿Qué riesgo evita? |
| --- | --- | --- |
| Estimador entrenado | **model.joblib** | ______________________________ |
| Versión del esquema | ______________________________ | ______________________________ |
| Orden y nombres de features | ______________________________ | ______________________________ |
| Versión del preprocesado | ______________________________ | ______________________________ |
| Etiquetas de salida permitidas | ______________________________ | ______________________________ |
| Versión del modelo | ______________________________ | ______________________________ |
| Tipo de estimador | ______________________________ | ______________________________ |

## 3. Práctica 2 — Construir un manifiesto mínimo

Completa el borrador de Python de la siguiente celda y ejecútalo para producir un JSON legible. Mantén los nombres de campo estables: el manifiesto es una interfaz entre quien empaqueta el modelo y quien lo ejecuta.

~~~json
{
  "schema_version": "",
  "model_version": "",
  "preprocessing_version": "",
  "feature_names": [],
  "output_labels": [],
  "estimator_type": ""
}
~~~

Comprueba antes de continuar que:

- aparecen exactamente las seis claves obligatorias;
- `feature_names` tiene once nombres y no incluye `sample_id`;
- `output_labels` representa las categorías que la interfaz podrá devolver.

Decide también qué debe pasar si:

1. falta **manifest.json**;
2. aparecen features desconocidas o en un orden distinto;
3. el modelo devuelve una etiqueta que no está en **output_labels**;
4. una fila del CSV no cumple el contrato de entrada.

In [ ]:
import json

feature_names = [
    "fixed_acidity",
    "volatile_acidity",
    "citric_acid",
    "residual_sugar",
    "chlorides",
    "free_sulfur_dioxide",
    "total_sulfur_dioxide",
    "density",
    "ph",
    "sulphates",
    "alcohol",
]

# No modifiques este orden: procede del contrato de la semana 3.
legacy_model_version = "wine-quality-rf-demo-v1"

manifest_draft = {
    "schema_version": "TODO",
    "model_version": legacy_model_version,
    "preprocessing_version": "TODO",
    "feature_names": feature_names,
    "output_labels": ["low", "medium", "high"],
    "estimator_type": "TODO",
}

required_keys = {
    "schema_version",
    "model_version",
    "preprocessing_version",
    "feature_names",
    "output_labels",
    "estimator_type",
}

assert set(manifest_draft) == required_keys
assert len(manifest_draft["feature_names"]) == 11
assert "sample_id" not in manifest_draft["feature_names"]

print(json.dumps(manifest_draft, indent=2, ensure_ascii=False))
print("\nLa estructura es válida. Sustituye los TODO y las etiquetas provisionales por decisiones justificadas.")

## 4. Práctica 3 — Anticipar fallos

Antes de ver la demo del profesorado, completa la tabla. No describas solo el error: indica en qué frontera debería detenerse el flujo.

| Cambio | Frontera que debe rechazarlo | Mensaje útil esperado | ¿Se escribe CSV? |
| --- | --- | --- | --- |
| Se intercambian `density` y `alcohol` | __________________ | __________________ | __________________ |
| Cambia `preprocessing_version` | __________________ | __________________ | __________________ |
| El modelo devuelve `unknown` | __________________ | __________________ | __________________ |
| La petición incluye `ph = 99` | __________________ | __________________ | __________________ |

## 5. Práctica 4 — Preparar la implementación de la segunda clase

Antes de pasar al taller, deja cerradas estas decisiones:

- ¿Qué campos son obligatorios y cuáles tienen valores por defecto?
- ¿Aceptarás campos desconocidos en el manifiesto?
- ¿Validarás el manifiesto antes de cargar el objeto serializado?
- ¿El CSV de salida se escribe fila a fila o solo después de validar todas las entradas?
- ¿Qué mensaje verá la persona que ejecuta la CLI cuando el contrato falle?
- ¿Qué ocurre si una segunda fila es inválida después de haber procesado una primera fila correcta?

La segunda clase convertirá estas decisiones en **ArtifactManifest**, **save_model_bundle**, **load_model_bundle**, inferencia y pruebas.